# Модель выявления аномалий (Isolation Forest)

**Вход:** `clean_final.csv` — очищенный датасет из `pipeline_final.ipynb` (25 002 записи с флагами аномалий по правилам).

**Идея:** правила ловят то, что мы придумали заранее. Модель может найти паттерны, о которых мы не подумали.

**Подход — semi-supervised:**
1. Берём только «чистые» записи (без аномалий по правилам) — модель учит, как выглядит **нормальная** запись
2. Предсказываем на **всём** датасете
3. Проверяем: ловит ли модель те же аномалии, что правила? Что нового нашла?

Это лучше, чем обучать на всём: модель не путает аномалии с нормой.

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Загружаем результат pipeline_final
df = pd.read_csv('clean_final.csv', parse_dates=['bdate', 'test_date', 'guard_bdate'])

print(f"Загружено: {len(df)} записей")
print(f"Уникальных детей: {df['person_id'].nunique()}")
print(f"Аномалий по правилам: {df['has_anomaly'].sum()} ({df['has_anomaly'].mean()*100:.1f}%)")
print(f"Чистых записей: {(~df['has_anomaly']).sum()}")

## 2. Feature Engineering

Из каждой записи извлекаем числовые признаки. Модель не понимает ФИО и даты — ей нужны числа.

**Признаки записи** (что можно вычислить из одной строки):
- `child_age` — возраст ребёнка на момент теста
- `parent_age_at_birth` — возраст родителя при рождении ребёнка
- `age_class_diff` — отклонение реального возраста от ожидаемого (класс + 6 лет)
- `id_eq_guard` — совпадает ли id ребёнка с id родителя (1/0)
- `same_school` — направляющая школа = площадка тестирования (1/0)
- `class_num` — номер класса

**Агрегаты по ребёнку** (нужно знать все записи этого person_id):
- `n_tests` — сколько раз тестировался всего
- `n_classes` — за сколько разных классов
- `min_gap` — минимальный интервал между тестами
- `max_class_jump` — максимальный скачок класса

In [ ]:
# --- Признаки из одной записи ---
df['class_num'] = pd.to_numeric(df['class'], errors='coerce')
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
df['parent_age_at_birth'] = (df['bdate'] - df['guard_bdate']).dt.days / 365.25
df['age_class_diff'] = df['child_age'] - (df['class_num'] + 6)
df['id_eq_guard'] = (df['id_doc'] == df['guard_id_doc']).astype(int)
df['same_school'] = (df['ogrn_naprav'] == df['ogrn_area']).astype(int)

# --- Агрегаты по person_id ---

# Количество тестов и классов
person_stats = df.groupby('person_id').agg(
    n_tests=('test_date', 'count'),
    n_classes=('class_num', 'nunique'),
).reset_index()

# Минимальный интервал между тестами
df_sorted = df.sort_values(['person_id', 'test_date'])
df_sorted['days_gap'] = (
    df_sorted['test_date'] - df_sorted.groupby('person_id')['test_date'].shift(1)
).dt.days
gap_stats = df_sorted.groupby('person_id')['days_gap'].agg(min_gap='min').reset_index()
gap_stats['min_gap'] = gap_stats['min_gap'].fillna(999)

# Максимальный скачок класса
def max_class_jump(group):
    classes = group.sort_values('test_date')['class_num'].dropna().values
    if len(classes) < 2:
        return 0
    return max(abs(classes[i] - classes[i-1]) for i in range(1, len(classes)))

jump_stats = df.groupby('person_id').apply(max_class_jump).reset_index()
jump_stats.columns = ['person_id', 'max_class_jump']

# Мёржим всё обратно
df = df.merge(person_stats, on='person_id', how='left')
df = df.merge(gap_stats, on='person_id', how='left')
df = df.merge(jump_stats, on='person_id', how='left')

# Итоговый набор фичей
feature_cols = [
    'child_age', 'parent_age_at_birth', 'age_class_diff',
    'id_eq_guard', 'same_school',
    'n_tests', 'n_classes', 'min_gap', 'max_class_jump', 'class_num'
]

X = df[feature_cols].fillna(0)
print(f"Матрица признаков: {X.shape}")
print(X.describe().round(2))

## 3. Разделение на train / test

**Semi-supervised подход:**
- **Train** — только чистые записи (`has_anomaly == False`). Модель учит «как выглядит нормальная запись».
- **Test** — весь датасет целиком. Модель оценивает каждую запись: похожа на нормальную или нет.

Дополнительно внутри train делаем 80/20 split для проверки стабильности.

In [ ]:
# Чистые записи — обучающая выборка
clean_mask = ~df['has_anomaly']
X_clean = X[clean_mask]

# 80/20 split внутри чистых для валидации
X_train, X_val = train_test_split(X_clean, test_size=0.2, random_state=42)

print(f"Чистых записей (обучение): {len(X_clean)}")
print(f"  Train: {len(X_train)}")
print(f"  Validation: {len(X_val)}")
print(f"Весь датасет (предсказание): {len(X)}")
print(f"  Из них аномалий по правилам: {df['has_anomaly'].sum()}")

## 4. Обучение Isolation Forest

**Как работает Isolation Forest:**
- Строит 200 случайных деревьев
- Каждое дерево случайно выбирает признак и точку разреза, делит данные пополам
- «Нормальная» запись окружена тысячами похожих → нужно много разрезов чтобы её изолировать
- «Аномальная» запись непохожа на остальные → изолируется за 2-3 разреза
- `anomaly_score` = среднее число разрезов. Чем меньше → тем аномальнее

Обучаем **только на чистых данных** (train), предсказываем на **всём датасете**.

In [ ]:
# Масштабирование: приводим все фичи к одному диапазону
scaler = StandardScaler()
scaler.fit(X_train)  # fit только на train!

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_all_scaled = scaler.transform(X)

# Обучение модели
model = IsolationForest(
    n_estimators=200,     # 200 деревьев
    contamination=0.05,   # ожидаем ~5% аномалий среди чистых (запас на ошибки правил)
    random_state=42
)
model.fit(X_train_scaled)  # обучаем ТОЛЬКО на чистых!

print("Модель обучена на чистых записях")

## 5. Валидация на hold-out

Проверяем на 20% чистых записей, которые модель не видела при обучении.
Если модель хорошая — почти все validation записи она должна считать нормальными.

In [ ]:
# Предсказание на validation (тоже чистые записи — модель не видела)
val_scores = model.decision_function(X_val_scaled)
val_preds = model.predict(X_val_scaled)
val_anomaly_rate = (val_preds == -1).mean()

print(f"Validation ({len(X_val)} чистых записей, модель их не видела):")
print(f"  Помечено как аномалия: {(val_preds == -1).sum()} ({val_anomaly_rate*100:.1f}%)")
print(f"  Помечено как норма: {(val_preds == 1).sum()} ({(val_preds == 1).mean()*100:.1f}%)")
print(f"")
if val_anomaly_rate < 0.10:
    print(f"  ✓ Модель стабильна: false positive rate {val_anomaly_rate*100:.1f}% на чистых данных")
else:
    print(f"  ⚠ Высокий false positive rate: {val_anomaly_rate*100:.1f}%")

## 6. Предсказание на всём датасете

In [ ]:
# Предсказание на ВСЕХ записях (и чистых, и аномальных по правилам)
df['anomaly_score'] = model.decision_function(X_all_scaled)
df['model_anomaly'] = model.predict(X_all_scaled) == -1

print(f"Результаты модели на всём датасете ({len(df)} записей):")
print(f"  Аномалий по модели: {df['model_anomaly'].sum()} ({df['model_anomaly'].mean()*100:.1f}%)")
print(f"  Нормальных по модели: {(~df['model_anomaly']).sum()}")

## 7. Сравнение: модель vs правила

Матрица пересечения — что нашли правила, что модель, что оба.

In [ ]:
both = (df['model_anomaly'] & df['has_anomaly']).sum()
model_only = (df['model_anomaly'] & ~df['has_anomaly']).sum()
rule_only = (~df['model_anomaly'] & df['has_anomaly']).sum()
neither = (~df['model_anomaly'] & ~df['has_anomaly']).sum()

print(f"{'':20} | Модель: ДА  | Модель: НЕТ")
print(f"{'-'*55}")
print(f"{'Правила: ДА':20} | {both:>10}  | {rule_only:>11}")
print(f"{'Правила: НЕТ':20} | {model_only:>10}  | {neither:>11}")
print()
print(f"Совпадение (оба):    {both}  — модель подтверждает правила")
print(f"Только модель:       {model_only}  — НОВЫЕ НАХОДКИ")
print(f"Только правила:      {rule_only}  — модель не считает аномалией")
print(f"Чистые (оба):        {neither}")

# Recall модели: какую долю аномалий по правилам модель поймала?
recall = both / (both + rule_only) * 100
print(f"\nRecall модели (доля пойманных аномалий правил): {recall:.1f}%")

## 8. Анализ новых находок модели

Смотрим: чем записи «только модель» отличаются от среднего?

In [ ]:
new_finds = df[df['model_anomaly'] & ~df['has_anomaly']].copy()

print(f"Новых находок модели: {len(new_finds)}")
print(f"\n{'Признак':25} | {'Новые':>10} | {'Всё':>10} | {'Разница':>10}")
print(f"{'-'*60}")

for col in feature_cols:
    m_new = new_finds[col].mean()
    m_all = df[col].mean()
    diff = m_new - m_all
    marker = ' ◄◄◄' if abs(diff) > 1 else ''
    print(f"{col:25} | {m_new:10.2f} | {m_all:10.2f} | {diff:+10.2f}{marker}")

In [ ]:
# Главная находка: same_school
pct_new = new_finds['same_school'].mean() * 100
pct_all = df['same_school'].mean() * 100

print(f"\n{'='*60}")
print(f"КЛЮЧЕВАЯ НАХОДКА: same_school (ogrn_naprav == ogrn_area)")
print(f"{'='*60}")
print(f"")
print(f"  В новых находках модели: {pct_new:.1f}%")
print(f"  Во всём датасете:        {pct_all:.1f}%")
print(f"  Превышение:              в {pct_new/max(pct_all,0.01):.1f} раз")
print(f"")
print(f"  Школа направляет ребёнка на тестирование")
print(f"  и сама же проводит это тестирование.")
print(f"  Потенциальный конфликт интересов.")

## 9. Важность признаков

Какие признаки сильнее всего влияют на решение модели?
Отрицательная корреляция = чем больше признак, тем аномальнее запись.

In [ ]:
correlations = df[feature_cols + ['anomaly_score']].corr()['anomaly_score'].drop('anomaly_score')

print(f"{'Признак':25} | {'Корреляция':>11} | Интерпретация")
print(f"{'-'*75}")

for feat in correlations.abs().sort_values(ascending=False).index:
    r = correlations[feat]
    if r < -0.1:
        interp = "↑ больше = аномальнее"
    elif r > 0.1:
        interp = "↑ больше = нормальнее"
    else:
        interp = "слабое влияние"
    print(f"{feat:25} | r = {r:+.3f}   | {interp}")

## 10. Топ самых аномальных записей

In [ ]:
top = df.nsmallest(15, 'anomaly_score')

print("Топ-15 самых аномальных записей по модели:\n")
for _, r in top.iterrows():
    rule_flag = "ПРАВИЛА+МОДЕЛЬ" if r['has_anomaly'] else "ТОЛЬКО МОДЕЛЬ"
    print(f"  [{rule_flag}] {r['last_name']} {r['first_name']} | "
          f"class={r['class']} | age={r['child_age']:.1f} | "
          f"tests={r['n_tests']} | classes={r['n_classes']} | "
          f"min_gap={r['min_gap']:.0f}d | "
          f"same_school={r['same_school']} | "
          f"score={r['anomaly_score']:.3f}")

## 11. Итого и сохранение

**Два слоя проверки:**
- **Правила** — ловят конкретные технические ошибки (частота, скачки классов, совпадение ID)
- **Модель** — находит записи «непохожие на нормальные» по совокупности признаков

**Ключевая находка модели:** паттерн `same_school` — школа сама направляет и сама тестирует. Правила это не проверяли.

In [ ]:
# Объединённый флаг
df['combined_anomaly'] = df['has_anomaly'] | df['model_anomaly']

print(f"Итоговая статистика:")
print(f"  Аномалий по правилам:  {df['has_anomaly'].sum()}")
print(f"  Аномалий по модели:    {df['model_anomaly'].sum()}")
print(f"  Объединённый флаг:     {df['combined_anomaly'].sum()} ({df['combined_anomaly'].mean()*100:.1f}%)")
print(f"  Чистых записей:        {(~df['combined_anomaly']).sum()}")

# Сохраняем
export_cols = [c for c in df.columns if c not in [
    'person_key', 'class_num', 'child_age', 'parent_age_at_birth'
]]
df[export_cols].to_csv('dataset_with_model_scores.csv', index=False)
print(f"\nСохранено: dataset_with_model_scores.csv")